In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch


dataset = load_dataset(
    "dim/hendrycks_math_train_1k_DeepSeek-R1-Distill-Qwen-1.5B_max_len_4096_greedy"
)
dataset = dataset["train"].train_test_split(
    # test_size=250,
    test_size=350,
    # test_size=999,
    # test_size=1,
    seed=42,
)
dataset = dataset["test"].filter(lambda x: x["model_answer"].count("</think>") == 1)

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    # attn_implementation="sdpa",
    attn_implementation="flash_attention_2",
)
model.requires_grad_(False)
tokenizer = AutoTokenizer.from_pretrained(model_name)

### Кодируем части текста в вектора

In [2]:
!pip install more_itertools -q

### Fixed train part

In [2]:
from more_itertools import chunked
from tqdm import tqdm
import itertools
import torch
import math

start_item = 100
cramming_tokens = []

for i in range(start_item, len(dataset)):
    print(f"{i}/{len(dataset)}")
    model_answer = dataset[i]["model_answer"]
    tokens = tokenizer.encode(
        dataset[i]["model_answer"],
        add_special_tokens=False,
    )
    mem_tokens = 8
    encode_size = mem_tokens * 4
    tokens_chunks = list(chunked(tokens, encode_size))
    max_parts = 10
    for chunk_part in range(2, max_parts):
        train_part = tokens_chunks[:chunk_part]
        train_part = list(itertools.chain(*train_part))

        train_part = torch.tensor(
            train_part,
            device="cuda",
        ).unsqueeze(0)

        target_len = encode_size + 1
        num_repeats = (target_len + mem_tokens - 1) // mem_tokens
        new_compression_len = num_repeats * mem_tokens

        context_len = train_part.shape[1] - encode_size
        if context_len < 0:
            continue

        labels_part = torch.full(
            (1, context_len + new_compression_len),
            -100,
            device="cuda",
            dtype=torch.long,
        )

        original_target_tokens = train_part[:, -encode_size:]

        new_target_labels = torch.cat(
            [
                original_target_tokens[:, 0].unsqueeze(1),
                original_target_tokens,
            ],
            dim=1,
        )

        labels_part[:, context_len : context_len + target_len] = new_target_labels

        compression_tensor_param = torch.nn.Parameter(
            torch.rand(
                mem_tokens,
                model.get_input_embeddings().weight.shape[1],
                device="cuda",
            ).unsqueeze(0),
            requires_grad=True,
        )
        optimizer = torch.optim.AdamW(
            [compression_tensor_param],
            lr=0.1,
        )

        prev_tokens = train_part[:, :-encode_size].clone()
        prev_embeds = model.get_input_embeddings()(prev_tokens)

        epoch_amount = 50
        dtype = torch.bfloat16
        for epoch in tqdm(range(epoch_amount)):
            compression_tensor = compression_tensor_param.repeat(1, num_repeats, 1)

            input_embeds = torch.cat(
                [
                    prev_embeds,
                    compression_tensor,
                ],
                dim=1,
            ).to(dtype)

            assert input_embeds.shape[1] == labels_part.shape[1]

            model_predicts = model(
                inputs_embeds=input_embeds,
                labels=labels_part,
            )

            compression_loss = model_predicts.loss
            compression_loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        # --- НАЧАЛО ИЗМЕНЕНИЙ В ЛОГИКЕ ПРОВЕРКИ ---

        # 1. Эталонная последовательность для проверки - это все метки, КРОМЕ ПОСЛЕДНЕЙ.
        #    Ее длина `encode_size`.
        check_target_labels = new_target_labels[0, 1:]
        check_target_str = tokenizer.decode(
            check_target_labels, add_special_tokens=False
        )

        # 2. Извлекаем предсказания модели. Нам нужна та же длина, что и у эталона.
        predicted_tokens = model_predicts.logits.argmax(-1)
        # Берем срез длиной `target_len`, как и раньше...
        predicted_target_part = predicted_tokens[
            :, context_len : context_len + target_len
        ][0]

        # 3. ...но для сравнения отбрасываем последний токен.
        prediction_str = tokenizer.decode(
            predicted_target_part[:-1], add_special_tokens=False
        )

        # 4. Сравниваем строки длиной `encode_size`.
        correct_reconstruction = prediction_str == check_target_str

        # --- КОНЕЦ ИЗМЕНЕНИЙ В ЛОГИКЕ ПРОВЕРКИ ---
        print(f"FULL TEXT: '{tokenizer.decode(train_part[:, :][-1])}'")
        print("-" * 50)
        print(
            f"Context: '{tokenizer.decode(train_part[:, :-encode_size][-1])}'",
        )
        # Эталонная строка, которую модель должна была сгенерировать
        print(
            f"TARGET to generate: '{check_target_str}'",
        )
        # Строка, которую модель сгенерировала на самом деле
        print(f"PREDICTED string:   '{prediction_str}'")
        print(f"Correct reconstruction: '{correct_reconstruction}'")
        print("=" * 50)
        print("=" * 50)
        cramming_tokens.append(compression_tensor_param.detach())
        # break

    break

100/209


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:01<00:00, 31.83it/s]


FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
--------------------------------------------------
Context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I'
TARGET to generate: ' remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
PREDICTED string:   ' remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
Correct reconstruction: 'True'


100%|██████████| 50/50 [00:01<00:00, 38.11it/s]


...

FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is'
--------------------------------------------------
Context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me thin

100%|██████████| 50/50 [00:01<00:00, 34.45it/s]


FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

But wait, is there a trick here? Maybe the'
--------------------------------------

100%|██████████| 50/50 [00:01<00:00, 33.50it/s]

FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

But wait, is there a trick here? Maybe the problem is trying to trick me into thin

### Decode part

In [3]:
import torch
from more_itertools import chunked
import itertools

# --- ПРЕДПОЛАГАЕТСЯ, ЧТО ЭТИ ОБЪЕКТЫ УЖЕ СУЩЕСТВУЮТ В ВАШЕЙ СРЕДЕ ---
# model: ваша языковая модель, уже загруженная на GPU
# tokenizer: ваш токенизатор
# dataset: ваш датасет
# cramming_tokens: список с обученными тензорами
# --------------------------------------------------------------------------


def predict_in_one_pass(
    model,
    tokenizer,
    context_tokens: torch.Tensor,
    learned_compression_tensor: torch.Tensor,
    expected_len: int,
):
    """
    Выполняет один прямой проход и извлекает предсказанные токены.
    Возвращает декодированный текст и список ID токенов.
    (Эта функция остается без изменений)
    """
    model.eval()
    device = model.device
    dtype = model.dtype

    context_tokens = context_tokens.to(device)
    mem_tokens = learned_compression_tensor.shape[1]
    target_len = expected_len + 1
    num_repeats = (target_len + mem_tokens - 1) // mem_tokens
    repeated_compression_tensor = learned_compression_tensor.repeat(
        1, num_repeats, 1
    ).to(dtype)

    with torch.no_grad():
        context_embeds = model.get_input_embeddings()(context_tokens)
        input_embeds = torch.cat(
            [context_embeds, repeated_compression_tensor],
            dim=1,
        )
        outputs = model(inputs_embeds=input_embeds)
        context_len = context_embeds.shape[1]
        predicted_logits = outputs.logits[
            :, context_len : context_len + expected_len, :
        ]
        predicted_token_ids = torch.argmax(predicted_logits, dim=-1)
        reconstructed_text = tokenizer.decode(
            predicted_token_ids[0], skip_special_tokens=True
        )

    return reconstructed_text, predicted_token_ids[0].tolist()


# --- ОСНОВНАЯ ЛОГИКА С ПОЛНЫМ ВЫВОДОМ И ОСТАНОВКОЙ ПРИ ОШИБКЕ ---

# Параметры
start_item = 100
mem_tokens = 8
encode_size = mem_tokens * 4
num_chunks_to_test = 5

print(
    f"Запускаем восстановление для {num_chunks_to_test} чанков С ОСТАНОВКОЙ ПРИ ПЕРВОЙ ОШИБКЕ..."
)
print("=" * 80)

# 1. Получаем исходные данные
item_index = start_item
model_answer = dataset[item_index]["model_answer"]
tokens = tokenizer.encode(model_answer, add_special_tokens=False)
tokens_chunks = list(chunked(tokens, encode_size))

# 2. Проверка данных
if len(tokens_chunks) < num_chunks_to_test + 1:
    raise ValueError(
        f"Недостаточно чанков в тексте ({len(tokens_chunks)}) для теста {num_chunks_to_test} чанков."
    )
if len(cramming_tokens) < num_chunks_to_test:
    raise ValueError(
        f"Недостаточно обученных тензоров ({len(cramming_tokens)}) для теста {num_chunks_to_test} чанков."
    )

# --- ДОБАВЛЕНО ДЛЯ ОТЛАДКИ: Вывод полного эталонного текста ---
total_chunks_to_process = 1 + num_chunks_to_test
original_tokens_goal = list(itertools.chain(*tokens_chunks[:total_chunks_to_process]))
print("--- Полный эталонный текст для восстановления ---")
print(tokenizer.decode(original_tokens_goal))
print("=" * 80)
# ---------------------------------------------------------------

# 3. Начинаем цикл восстановления
current_context_tokens = list(tokens_chunks[0])
total_correct_chunks = 0
last_processed_chunk_index = -1

for i in range(num_chunks_to_test):
    last_processed_chunk_index = i
    print(f"\n--- Восстановление чанка {i+1}/{num_chunks_to_test} ---")

    # a. Подготовка данных
    context_tensor = torch.tensor([current_context_tokens], dtype=torch.long)
    learned_tensor = cramming_tokens[i]
    original_target_chunk = tokens_chunks[i + 1]
    original_target_text = tokenizer.decode(original_target_chunk)

    # --- ИЗМЕНЕНО ДЛЯ ОТЛАДКИ: Полный вывод контекста ---
    print(
        f"Текущий контекст (полный восстановленный текст, {len(current_context_tokens)} токенов):"
    )
    print(f"'{tokenizer.decode(current_context_tokens)}'")
    print("-" * 40)
    # ----------------------------------------------------

    # b. Инференс
    reconstructed_text, reconstructed_token_ids = predict_in_one_pass(
        model=model,
        tokenizer=tokenizer,
        context_tokens=context_tensor,
        learned_compression_tensor=learned_tensor,
        expected_len=encode_size,
    )

    # c. Вывод результатов шага
    print(f"ЭТАЛОН для этого шага:    '{original_target_text}'")
    print(f"РЕЗУЛЬТАТ этого шага: '{reconstructed_text}'")

    # d. Проверка и обновление контекста или остановка цикла
    if original_target_text.strip() == reconstructed_text.strip():
        print("✅ Результат верный.")
        total_correct_chunks += 1
        current_context_tokens.extend(reconstructed_token_ids)
    else:
        print("❌ ОШИБКА в восстановлении.")
        print(
            "Останавливаю дальнейшее восстановление, так как контекст был бы искажен."
        )
        current_context_tokens.extend(reconstructed_token_ids)
        break

# --- Итоговые результаты ---
print("\n" + "=" * 80)
print("             ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("=" * 80)
print(
    f"Успешно восстановлено: {total_correct_chunks} из {last_processed_chunk_index + 1} предпринятых попыток."
)

# Собираем эталонный текст той же длины, что и восстановленный, для прямого сравнения
# Он будет состоять из 1 начального чанка и `last_processed_chunk_index + 1` целевых чанков
num_original_chunks_to_compare = 1 + last_processed_chunk_index + 1
original_tokens_to_compare = list(
    itertools.chain(*tokens_chunks[:num_original_chunks_to_compare])
)
original_full_text = tokenizer.decode(original_tokens_to_compare)

reconstructed_full_text = tokenizer.decode(current_context_tokens)

print("\n--- Оригинальный текст (до точки остановки) ---")
print(original_full_text)
print("\n--- Восстановленный текст (до точки остановки) ---")
print(reconstructed_full_text)
print("=" * 80)

Запускаем восстановление для 5 чанков С ОСТАНОВКОЙ ПРИ ПЕРВОЙ ОШИБКЕ...
--- Полный эталонный текст для восстановления ---
Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1

--- Восстановление чанка 1/5 ---
Текущий контекст (полный восстановленный текст, 32 токенов):
'Okay

### Проверка пространства

### Интерполяция пространства

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
from tqdm import tqdm
import itertools
from more_itertools import chunked
import math

# --- 1. НАСТРОЙКА ---
device = "cuda" if torch.cuda.is_available() else "cpu"
# Для моделей такого размера может потребоваться GPU с > 8GB VRAM
# Если возникает ошибка OOM, попробуйте модель поменьше
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"Используется устройство: {device}")

# Загрузка модели и токенизатора
print(f"Загрузка модели '{model_name}'...")
# Используем bfloat16 для экономии памяти и ускорения, если доступно
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float32
)

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=dtype, trust_remote_code=True
    ).to(device)
    print("Модель успешно загружена.")
except Exception as e:
    print(f"Ошибка при загрузке модели: {e}")
    print(
        "Убедитесь, что у вас установлены все зависимости (pip install transformers torch accelerate) и есть доступ в интернет."
    )
    exit()


# Добавляем паддинг токен, если его нет. Это хорошая практика.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# --- 2. ФУНКЦИЯ "СЖАТИЯ" ТЕКСТА В ВЕКТОР ---


def cram_text_to_vector(
    text_to_cram: str, mem_tokens: int = 8, epochs: int = 100, lr: float = 0.1
):
    """
    Принимает строку текста, сжимает её в вектор z и возвращает его.
    Это ваш код, обернутый в функцию.
    """
    print(f"\n--- Сжатие текста: '{text_to_cram[:50]}...' ---")

    tokens = tokenizer.encode(text_to_cram, add_special_tokens=False)

    # Мы будем сжимать весь текст в один блок
    target_tokens = torch.tensor(tokens, device=device).unsqueeze(0)
    target_len = target_tokens.shape[1]

    if target_len == 0:
        print("Ошибка: текст не содержит токенов.")
        return None

    # Вычисляем, сколько раз нужно повторить mem_tokens, чтобы покрыть target_len
    num_repeats = (target_len + mem_tokens - 1) // mem_tokens
    input_len = num_repeats * mem_tokens

    # Создаем метки (labels) для обучения
    labels = torch.full((1, input_len), -100, device=device, dtype=torch.long)
    # Модель должна предсказать target_tokens[i] увидев на входе i-1 токенов
    labels[:, :target_len] = target_tokens

    # Инициализируем наш сжимающий тензор z
    compression_tensor_param = torch.nn.Parameter(
        torch.randn(  # Используем randn для лучшей начальной инициализации
            1,
            mem_tokens,
            model.get_input_embeddings().weight.shape[1],
            device=device,
        )
        * 0.01,  # Маленькая дисперсия на старте
        requires_grad=True,
    )

    optimizer = torch.optim.AdamW([compression_tensor_param], lr=lr)

    pbar = tqdm(range(epochs), desc="Epochs")
    for epoch in pbar:
        optimizer.zero_grad()

        # Повторяем наш z-вектор, чтобы соответствовать длине входа
        input_embeds = compression_tensor_param.repeat(1, num_repeats, 1).to(dtype)

        assert input_embeds.shape[1] == labels.shape[1]

        outputs = model(inputs_embeds=input_embeds, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    print(f"--- Сжатие завершено. Финальная потеря (loss): {loss.item():.4f} ---")
    # Возвращаем только уникальный, не повторенный тензор
    return compression_tensor_param.detach()


# --- 3. ФУНКЦИЯ ИНТЕРПОЛЯЦИИ И "РАСПАКОВКИ" ---


def interpolate_and_unpack(
    z_start_vec: torch.Tensor,
    z_end_vec: torch.Tensor,
    steps: int = 5,
    unpack_len_tokens: int = 50,
):
    """
    Интерполирует между z-векторами и мгновенно "распаковывает" их в текст.
    """
    print("\n\n" + "=" * 20 + " НАЧАЛО ИНТЕРПОЛЯЦИИ И РАСПАКОВКИ " + "=" * 20)

    mem_tokens = z_start_vec.shape[1]
    num_repeats = (unpack_len_tokens + mem_tokens - 1) // mem_tokens

    for i in range(steps):
        alpha = i / (steps - 1) if steps > 1 else 0.0
        print(f"\n--- Шаг {i+1}/{steps} (alpha={alpha:.2f}) ---")

        # Линейная интерполяция (LERP)
        z_interp = torch.lerp(z_start_vec, z_end_vec, alpha)

        # Повторяем интерполированный вектор, чтобы создать полный вход
        input_embeds = z_interp.repeat(1, num_repeats, 1).to(dtype)

        # Выполняем ОДИН forward pass, чтобы получить логиты
        with torch.no_grad():
            outputs = model(inputs_embeds=input_embeds)
            logits = outputs.logits

        # Находим предсказанные токены, взяв argmax по логитам
        predicted_token_ids = torch.argmax(logits, dim=-1)

        # Обрезаем до нужной длины
        predicted_token_ids = predicted_token_ids[:, :unpack_len_tokens]

        # Декодируем и печатаем результат
        unpacked_text = tokenizer.decode(
            predicted_token_ids[0], skip_special_tokens=True
        )

        print(f"Распакованный текст: {unpacked_text}")

    print("\n" + "=" * 20 + " КОНЕЦ ИНТЕРПОЛЯЦИИ И РАСПАКОВКИ " + "=" * 20)


# --- 4. ОСНОВНОЙ БЛОК: ЗАПУСК ЭКСПЕРИМЕНТА ---

if __name__ == "__main__":
    # Тексты для интерполяции
    # Выбираем два семантически разных, но структурно похожих текста
    text_start = "The sun rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth."
    text_end = "The moon hangs high in the dark sky. Stars glitter like tiny diamonds. A silent night falls with cool air and shadows."

    print("Текст 1:", text_start)
    print("Текст 2:", text_end)

    # Сжимаем оба текста
    z_start = cram_text_to_vector(text_start)
    z_end = cram_text_to_vector(text_end)

    if z_start is not None and z_end is not None:
        # Вычисляем максимальную длину для распаковки
        max_len = max(
            len(tokenizer.encode(text_start)), len(tokenizer.encode(text_end))
        )

        # Запускаем интерполяцию
        interpolate_and_unpack(
            z_start,
            z_end,
            steps=105,
            unpack_len_tokens=max_len + 5,  # +5 токенов запаса
        )
    else:
        print("Один из векторов не был создан. Эксперимент прерван.")

### более продвинутые методы интерполяции

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
from tqdm import tqdm
import itertools
from more_itertools import chunked
import math
import numpy as np
from sklearn.decomposition import PCA

# --- БЛОК 1: НАСТРОЙКА ---
# ... (без изменений, как в предыдущем коде) ...
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"Используется устройство: {device}")
print(f"Загрузка модели '{model_name}'...")
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float32
)

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=dtype, trust_remote_code=True
    ).to(device)
    print("Модель успешно загружена.")
except Exception as e:
    print(f"Ошибка при загрузке модели: {e}")
    exit()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id


# --- БЛОК 2: ФУНКЦИЯ "СЖАТИЯ" ---
# ... (без изменений) ...
def cram_text_to_vector(
    text_to_cram: str, mem_tokens: int = 8, epochs: int = 100, lr: float = 0.1
):
    print(f"\n--- Сжатие текста: '{text_to_cram[:50]}...' ---")
    tokens = tokenizer.encode(text_to_cram, add_special_tokens=False)
    target_tokens = torch.tensor(tokens, device=device).unsqueeze(0)
    target_len = target_tokens.shape[1]
    if target_len == 0:
        print("Ошибка: текст не содержит токенов.")
        return None
    num_repeats = (target_len + mem_tokens - 1) // mem_tokens
    input_len = num_repeats * mem_tokens
    labels = torch.full((1, input_len), -100, device=device, dtype=torch.long)
    labels[:, :target_len] = target_tokens
    compression_tensor_param = torch.nn.Parameter(
        torch.randn(
            1, mem_tokens, model.get_input_embeddings().weight.shape[1], device=device
        )
        * 0.01,
        requires_grad=True,
    )
    optimizer = torch.optim.AdamW([compression_tensor_param], lr=lr)
    pbar = tqdm(range(epochs), desc="Epochs")
    for epoch in pbar:
        optimizer.zero_grad()
        input_embeds = compression_tensor_param.repeat(1, num_repeats, 1).to(dtype)
        assert input_embeds.shape[1] == labels.shape[1]
        outputs = model(inputs_embeds=input_embeds, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    print(f"--- Сжатие завершено. Финальная потеря (loss): {loss.item():.4f} ---")
    return compression_tensor_param.detach()


# --- НОВЫЙ БЛОК: ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---


def slerp(v0, v1, t, DOT_THRESHOLD=0.9995):
    """Сферическая линейная интерполяция"""
    v0_cpu = v0.cpu().numpy().flatten()
    v1_cpu = v1.cpu().numpy().flatten()
    t = float(t)

    dot = np.sum(v0_cpu * v1_cpu)
    if np.abs(dot) > DOT_THRESHOLD:
        # Если векторы почти коллинеарны, используем линейную интерполяцию
        result = (1 - t) * v0_cpu + t * v1_cpu
    else:
        # Угол между векторами
        theta = np.arccos(dot)
        sin_theta = np.sin(theta)
        # Формула SLERP
        result = (np.sin((1 - t) * theta) / sin_theta) * v0_cpu + (
            np.sin(t * theta) / sin_theta
        ) * v1_cpu

    # Возвращаем результат в исходной форме и на нужном устройстве
    return torch.from_numpy(result).to(v0.device).reshape(v0.shape)


def unpack_vector(z_vec, unpack_len_tokens):
    """Оборачиваем логику распаковки в отдельную функцию"""
    mem_tokens = z_vec.shape[1]
    num_repeats = (unpack_len_tokens + mem_tokens - 1) // mem_tokens
    input_embeds = z_vec.repeat(1, num_repeats, 1).to(dtype)
    with torch.no_grad():
        outputs = model(inputs_embeds=input_embeds)
        logits = outputs.logits
    predicted_token_ids = torch.argmax(logits, dim=-1)[:, :unpack_len_tokens]
    unpacked_text = tokenizer.decode(predicted_token_ids[0], skip_special_tokens=True)
    return unpacked_text


# --- ОБНОВЛЕННЫЙ БЛОК: ФУНКЦИИ ЭКСПЕРИМЕНТОВ ---


def run_interpolation_experiment(
    z_start, z_end, steps, unpack_len, interpolation_method="lerp"
):
    """
    Универсальная функция для проведения экспериментов с разными методами.
    """
    method_name = interpolation_method.upper()
    print("\n\n" + "=" * 20 + f" НАЧАЛО ИНТЕРПОЛЯЦИИ ({method_name}) " + "=" * 20)

    # Для PCA нам нужен средний вектор и главная компонента
    pca_mean = None
    pca_component = None
    if method_name == "PCA":
        # Сначала "расплющим" наши z векторы для PCA
        z_start_flat = z_start.reshape(1, -1)
        z_end_flat = z_end.reshape(1, -1)

        # Объединяем их в один датасет
        z_data = torch.cat([z_start_flat, z_end_flat], dim=0).cpu().numpy()

        pca = PCA(n_components=1)
        pca.fit(z_data)

        # Средняя точка между нашими векторами
        pca_mean = torch.from_numpy(pca.mean_).to(device).reshape(z_start.shape)
        # Направление главной оси вариации
        pca_component = (
            torch.from_numpy(pca.components_[0]).to(device).reshape(z_start.shape)
        )

    for i in range(steps):
        # Alpha от 0 до 1, а для PCA - от -X до +X
        alpha = i / (steps - 1) if steps > 1 else 0.0

        if method_name == "LERP":
            z_interp = torch.lerp(z_start, z_end, alpha)
        elif method_name == "SLERP":
            # Нормализуем векторы для корректной работы SLERP
            z_start_norm = z_start / torch.norm(z_start)
            z_end_norm = z_end / torch.norm(z_end)
            z_interp = slerp(z_start_norm, z_end_norm, alpha)
        elif method_name == "PCA":
            # Двигаемся от -1.5 до +1.5 стандартных отклонений вдоль главной компоненты
            # Это эмпирический диапазон, который обычно хорошо работает
            t = -1.5 + 3.0 * alpha
            z_interp = pca_mean + t * pca_component
            print(f"\n--- Шаг {i+1}/{steps} (alpha={alpha:.2f}, t={t:.2f}) ---")

        if method_name != "PCA":
            print(f"\n--- Шаг {i+1}/{steps} (alpha={alpha:.2f}) ---")

        unpacked_text = unpack_vector(z_interp, unpack_len)
        print(f"Распакованный текст ({method_name}): {unpacked_text}")

    print("\n" + "=" * 20 + f" КОНЕЦ ИНТЕРПОЛЯЦИИ ({method_name}) " + "=" * 20)


# --- ОСНОВНОЙ БЛОК: ЗАПУСК ЭКСПЕРИМЕНТОВ ---

if __name__ == "__main__":
    text_start = "The sun rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth."
    text_end = "The moon hangs high in the dark sky. Stars glitter like tiny diamonds. A silent night falls with cool air and shadows."

    print("Текст 1:", text_start)
    print("Текст 2:", text_end)

    # Сжимаем оба текста
    z_start = cram_text_to_vector(text_start)
    z_end = cram_text_to_vector(text_end)

    if z_start is not None and z_end is not None:
        max_len = max(
            len(tokenizer.encode(text_start)), len(tokenizer.encode(text_end))
        )
        unpack_len = max_len + 5
        steps = 110  # Возьмем 11 шагов для большей детализации

        # Эксперимент 1: LERP (для сравнения)
        run_interpolation_experiment(z_start, z_end, steps, unpack_len, "lerp")

        # Эксперимент 2: SLERP
        run_interpolation_experiment(z_start, z_end, steps, unpack_len, "slerp")

        # Эксперимент 3: PCA
        run_interpolation_experiment(z_start, z_end, steps, unpack_len, "pca")
    else:
        print("Один из векторов не был создан. Эксперимент прерван.")

Используется устройство: cuda
Загрузка модели 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'...
Модель успешно загружена.
Текст 1: The sun rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth.
Текст 2: The moon hangs high in the dark sky. Stars glitter like tiny diamonds. A silent night falls with cool air and shadows.

--- Сжатие текста: 'The sun rises over the green hills. Birds start to...' ---


Epochs: 100%|██████████| 100/100 [00:03<00:00, 26.76it/s, loss=0.3928]


--- Сжатие завершено. Финальная потеря (loss): 0.3928 ---

--- Сжатие текста: 'The moon hangs high in the dark sky. Stars glitter...' ---


Epochs: 100%|██████████| 100/100 [00:03<00:00, 28.56it/s, loss=0.5205]


--- Сжатие завершено. Финальная потеря (loss): 0.5205 ---


==================== НАЧАЛО ИНТЕРПОЛЯЦИИ (LERP) ====================

--- Шаг 1/110 (alpha=0.00) ---
Распакованный текст (LERP):  ensuring rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth. day incon with bright... and and

--- Шаг 2/110 (alpha=0.01) ---
Распакованный текст (LERP):  ensuring rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth. day incon with bright... and and

--- Шаг 3/110 (alpha=0.02) ---
Распакованный текст (LERP):  ensuring rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth. dayINF with bright... and and

--- Шаг 4/110 (alpha=0.03) ---
Распакованный текст (LERP):  ensuring rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth. dayINF with bright... and and



### Идеи которые не сработают(flow matching)

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
from tqdm import tqdm
import math

# --- БЛОКИ 1 и 2: НАСТРОЙКА и CRAM_TEXT_TO_VECTOR ---
# ... (вставьте сюда код из предыдущих ответов, он не меняется) ...
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"Используется устройство: {device}")
print(f"Загрузка модели '{model_name}'...")
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float32
)

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=dtype, trust_remote_code=True
    ).to(device)
    print("Модель успешно загружена.")
except Exception as e:
    print(f"Ошибка при загрузке модели: {e}")
    exit()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id


def cram_text_to_vector(
    text_to_cram: str, mem_tokens: int = 8, epochs: int = 100, lr: float = 0.1
):
    print(f"\n--- Сжатие текста: '{text_to_cram[:50]}...' ---")
    tokens = tokenizer.encode(text_to_cram, add_special_tokens=False)
    target_tokens = torch.tensor(tokens, device=device).unsqueeze(0)
    target_len = target_tokens.shape[1]
    if target_len == 0:
        print("Ошибка: текст не содержит токенов.")
        return None
    num_repeats = (target_len + mem_tokens - 1) // mem_tokens
    input_len = num_repeats * mem_tokens
    labels = torch.full((1, input_len), -100, device=device, dtype=torch.long)
    labels[:, :target_len] = target_tokens
    compression_tensor_param = torch.nn.Parameter(
        torch.randn(
            1, mem_tokens, model.get_input_embeddings().weight.shape[1], device=device
        )
        * 0.01,
        requires_grad=True,
    )
    optimizer = torch.optim.AdamW([compression_tensor_param], lr=lr)
    pbar = tqdm(range(epochs), desc="Epochs")
    for epoch in pbar:
        optimizer.zero_grad()
        input_embeds = compression_tensor_param.repeat(1, num_repeats, 1).to(dtype)
        assert input_embeds.shape[1] == labels.shape[1]
        outputs = model(inputs_embeds=input_embeds, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    print(f"--- Сжатие завершено. Финальная потеря (loss): {loss.item():.4f} ---")
    return compression_tensor_param.detach()


def unpack_vector(z_vec, unpack_len_tokens):
    """Оборачиваем логику распаковки в отдельную функцию"""
    mem_tokens = z_vec.shape[1]
    num_repeats = (unpack_len_tokens + mem_tokens - 1) // mem_tokens
    input_embeds = z_vec.repeat(1, num_repeats, 1).to(dtype)
    with torch.no_grad():
        outputs = model(inputs_embeds=input_embeds)
        logits = outputs.logits
    predicted_token_ids = torch.argmax(logits, dim=-1)[:, :unpack_len_tokens]
    unpacked_text = tokenizer.decode(predicted_token_ids[0], skip_special_tokens=True)
    return unpacked_text


# --- НОВЫЙ БЛОК: ОБУЧЕНИЕ ПУТИ (PATH FINDING) ---


def find_and_unpack_path(
    z_start,
    z_end,
    steps,
    unpack_len,
    train_epochs=50,
    lr=0.01,
    lambda_smooth=1.0,  # Коэффициент для плавности
):
    """
    Обучает путь между z_start и z_end и затем распаковывает точки на этом пути.
    """
    print("\n\n" + "=" * 20 + " НАЧАЛО ОБУЧЕНИЯ ПУТИ (PATH FINDING) " + "=" * 20)

    # 1. Инициализация пути
    # Создаем `steps-2` промежуточных точек и делаем их обучаемыми
    path_points = []
    for i in range(1, steps - 1):
        alpha = i / (steps - 1)
        # Инициализируем линейной интерполяцией
        point = torch.lerp(z_start, z_end, alpha).clone().detach().requires_grad_(True)
        path_points.append(point)

    # Полный путь включает фиксированные начальную и конечную точки
    full_path = [z_start] + path_points + [z_end]

    # Оптимизатор будет обновлять только промежуточные точки
    optimizer = torch.optim.AdamW(path_points, lr=lr)

    # 2. Цикл обучения пути
    pbar = tqdm(range(train_epochs), desc="Training Path")
    for epoch in pbar:
        optimizer.zero_grad()

        total_fidelity_loss = 0
        total_smoothness_loss = 0

        # Пересобираем полный путь на каждой итерации
        current_full_path = [z_start] + path_points + [z_end]

        for i in range(1, steps - 1):  # Итерируемся только по обучаемым точкам
            z_t = current_full_path[i]

            # --- Fidelity Loss (Потеря на осмысленность) ---
            # Распаковываем логиты и считаем их энтропию
            mem_tokens = z_t.shape[1]
            num_repeats = (unpack_len + mem_tokens - 1) // mem_tokens
            input_embeds = z_t.repeat(1, num_repeats, 1).to(dtype)

            outputs = model(inputs_embeds=input_embeds)
            logits = outputs.logits

            # Энтропия распределения вероятностей
            probs = F.softmax(logits, dim=-1)
            log_probs = F.log_softmax(logits, dim=-1)
            # Мы хотим минимизировать энтропию, чтобы модель была "уверенной"
            entropy = -torch.sum(probs * log_probs, dim=-1).mean()
            total_fidelity_loss += entropy

            # --- Smoothness Loss (Потеря на плавность) ---
            # Расстояние до предыдущей точки
            prev_point = current_full_path[i - 1]
            # Расстояние до следующей точки
            next_point = current_full_path[i + 1]
            # Штрафуем за резкие "изломы" пути
            # (z_t - prev) - (next - z_t) = 2*z_t - prev - next
            smoothness = torch.norm(2 * z_t - prev_point - next_point)
            total_smoothness_loss += smoothness

        total_loss = total_fidelity_loss + lambda_smooth * total_smoothness_loss
        total_loss.backward()
        optimizer.step()

        pbar.set_postfix(
            {
                "total_loss": f"{total_loss.item():.4f}",
                "fidelity": f"{total_fidelity_loss.item():.4f}",
                "smooth": f"{total_smoothness_loss.item():.4f}",
            }
        )

    print("--- Обучение пути завершено. ---")

    # 3. Распаковка точек с найденного пути
    print("\n" + "=" * 20 + " РАСПАКОВКА НАЙДЕННОГО ПУТИ " + "=" * 20)
    final_path = [z_start] + path_points + [z_end]
    for i, z_point in enumerate(final_path):
        alpha = i / (steps - 1)
        print(f"\n--- Шаг {i+1}/{steps} (alpha={alpha:.2f}) ---")
        unpacked_text = unpack_vector(z_point, unpack_len)
        print(f"Распакованный текст: {unpacked_text}")

    print("\n" + "=" * 20 + " КОНЕЦ РАСПАКОВКИ ПУТИ " + "=" * 20)


# --- ОСНОВНОЙ БЛОК: ЗАПУСК ЭКСПЕРИМЕНТА ---

if __name__ == "__main__":
    text_start = "The sun rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth."
    text_end = "The moon hangs high in the dark sky. Stars glitter like tiny diamonds. A silent night falls with cool air and shadows."

    print("Текст 1:", text_start)
    print("Текст 2:", text_end)

    # Сжимаем оба текста
    z_start = cram_text_to_vector(text_start)
    z_end = cram_text_to_vector(text_end)

    if z_start is not None and z_end is not None:
        max_len = max(
            len(tokenizer.encode(text_start)), len(tokenizer.encode(text_end))
        )
        unpack_len = max_len + 5
        steps = 50

        # Запускаем эксперимент по поиску пути
        find_and_unpack_path(
            z_start,
            z_end,
            steps=steps,
            unpack_len=unpack_len,
            train_epochs=100,  # Количество эпох для обучения пути
            lr=0.01,
            lambda_smooth=10.0,  # Возможно, понадобится потюнить этот коэффициент
        )
    else:
        print("Один из векторов не был создан. Эксперимент прерван.")

Используется устройство: cuda
Загрузка модели 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'...
Модель успешно загружена.
Текст 1: The sun rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth.
Текст 2: The moon hangs high in the dark sky. Stars glitter like tiny diamonds. A silent night falls with cool air and shadows.

--- Сжатие текста: 'The sun rises over the green hills. Birds start to...' ---


Epochs: 100%|██████████| 100/100 [00:03<00:00, 26.64it/s, loss=0.0305]


--- Сжатие завершено. Финальная потеря (loss): 0.0305 ---

--- Сжатие текста: 'The moon hangs high in the dark sky. Stars glitter...' ---


Epochs: 100%|██████████| 100/100 [00:03<00:00, 28.38it/s, loss=0.7050]


--- Сжатие завершено. Финальная потеря (loss): 0.7050 ---


==================== НАЧАЛО ОБУЧЕНИЯ ПУТИ (PATH FINDING) ====================


Training Path: 100%|██████████| 100/100 [02:59<00:00,  1.80s/it, total_loss=140.4415, fidelity=58.7500, smooth=8.1691]  


--- Обучение пути завершено. ---

==================== РАСПАКОВКА НАЙДЕННОГО ПУТИ ====================

--- Шаг 1/50 (alpha=0.00) ---
Распакованный текст:  sun rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth. begins with the light. words

--- Шаг 2/50 (alpha=0.02) ---
Распакованный текст:  sun rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth. takes with morning.. starts

--- Шаг 3/50 (alpha=0.04) ---
Распакованный текст:  sun rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth. takes with morning.. starts

--- Шаг 4/50 (alpha=0.06) ---
Распакованный текст:  sun rises over the green hills. Birds start to sing their morning songs. A new day begins with bright light and warmth. takes with morning.. starts

--- Шаг 5/50 (alpha=0.08) ---
Распакованный текст:  sun rises over the green hills. Birds s